# Functional Propagation Preparation

This notebook contains code to generate a GAF file from the GO annotations from the three GO annotation methods.

In [2]:
from Bio.UniProt.GOA import GAF20FIELDS
from datetime import datetime
from hogprop.OBOParser import OBO
from tqdm.auto import tqdm
import os
import pandas as pd

In [3]:
go = OBO('../data/geneontology/go.obo', store_as_int=True)

## Read InterProScan GO results

In [3]:
# load InterPro GO results
ipr_go_df = pd.read_csv('../data/functional_predictions/interproscan/Tomato_dataset.with_GO.txt.gz', sep='\t', names=['prot_id', 'go_terms'])

In [4]:
def convert_long(df, prot_key, go_key, sep):
    def do():
        for (_, r) in tqdm(df.iterrows(), total=len(df)):
            for t in r[go_key].split(sep):
                yield (r[prot_key], t)
    return pd.DataFrame(do(), columns=['prot_id', 'go_id'])

In [5]:
ipr_go_df1 = convert_long(ipr_go_df, prot_key='prot_id', go_key='go_terms', sep=';')

  0%|          | 0/1444806 [00:00<?, ?it/s]

In [6]:
ipr_go_df1['source'] = 'InterProScan'

## Read EggNOG-Mapper results

In [7]:
eggnog_mapper_header = ['query',
 'seed_ortholog',
 'evalue',
 'score',
 'eggNOG_OGs',
 'max_annot_lvl',
 'COG_category',
 'Description',
 'Preferred_name',
 'GOs',
 'EC',
 'KEGG_ko',
 'KEGG_Pathway',
 'KEGG_Module',
 'KEGG_Reaction',
 'KEGG_rclass',
 'BRITE',
 'KEGG_TC',
 'CAZy',
 'BiGG_Reaction',
 'PFAMs']

def read_eggnog():
    def read(sp):
        fn = '../data/functional_predictions/eggnog-mapper/' + sp + '/' + sp + '.emapper.annotations.gz'
        df = pd.read_csv(fn, sep='\t', comment='#', names=eggnog_mapper_header)
        return df[['query', 'GOs']]
    def read_all():
        for sp in tqdm(os.listdir('../data/functional_predictions/eggnog-mapper')):
            yield read(sp)

    return pd.concat(read_all())

eggnog_df = read_eggnog()

  0%|          | 0/64 [00:00<?, ?it/s]

In [8]:
eggnog_df1 = convert_long(eggnog_df, prot_key='query', go_key='GOs', sep=',')

  0%|          | 0/2089595 [00:00<?, ?it/s]

In [9]:
eggnog_df1['source'] = 'eggnog-mapper'

In [10]:
eggnog_df1 = eggnog_df1[eggnog_df1['go_id'] != '-']

In [11]:
eggnog_df1.head()

,prot_id,go_id,source
0,TS204Solyc_TS204_01T000001.1,GO:0003674,eggnog-mapper
1,TS204Solyc_TS204_01T000001.1,GO:0003824,eggnog-mapper
2,TS204Solyc_TS204_01T000001.1,GO:0004351,eggnog-mapper
3,TS204Solyc_TS204_01T000001.1,GO:0005488,eggnog-mapper
4,TS204Solyc_TS204_01T000001.1,GO:0005515,eggnog-mapper


## Read FANTASIA-lite results

In [12]:
def read_fantasia():
    def read(sp):
        fn = '../data/functional_predictions/fantasia/results/' + sp + '/results_' + sp + '.csv.gz'
        df = pd.read_csv(fn)
        return df
    def read_all():
        for sp in tqdm(os.listdir('../data/functional_predictions/fantasia/results')):
            yield read(sp)

    return pd.concat(read_all())

fantasia_df = read_fantasia()

  0%|          | 0/64 [00:00<?, ?it/s]

In [13]:
fantasia_df = fantasia_df[['query_accession', 'go_id']].rename(columns={'query_accession': 'prot_id'})
fantasia_df['source'] = 'fantasia'

In [14]:
fantasia_df.head()

,prot_id,go_id,source
0,TS204Solyc_TS204_10T000148.1,GO:0005704,fantasia
1,TS204Solyc_TS204_07T000433.1,GO:0000398,fantasia
2,TS204Solyc_TS204_07T000433.1,GO:0000380,fantasia
3,TS204Solyc_TS204_05T002171.1,GO:0010118,fantasia
4,TS204Solyc_TS204_05T002171.1,GO:0072583,fantasia


In [15]:
df = pd.concat((ipr_go_df1, eggnog_df1, fantasia_df))

Remove obsolete GO terms from the annotations

In [16]:
ts = set(df.go_id)
obsolete_terms = set()
for t in ts:
    if t not in go:
        obsolete_terms.add(t)

df = df[~df.go_id.isin(obsolete_terms)]

In [17]:
df.head()

,prot_id,go_id,source
0,ARATHAT1G01010.1,GO:0006355,InterProScan
1,ARATHAT1G01010.1,GO:0003677,InterProScan
2,ARATHAT1G01020.1,GO:0005794,InterProScan
3,ARATHAT1G01020.1,GO:0006665,InterProScan
4,ARATHAT1G01020.1,GO:0016125,InterProScan


In [18]:
df["DB"] = df['source']
df["DB_Object_ID"] = df['prot_id']
# 3R DB Object Symbol
df["DB_Object_Symbol"] = df["DB_Object_ID"]
# 4O Qualifier
df["Qualifier"] = ""
# 5R GO ID
#df["GO_ID"] = df["TermNr"].apply(lambda t: "GO:{:07d}".format(t))
df['GO_ID'] = df['go_id']
# 6R DB:Reference
df["DB:Reference"] = df['source']
# 7R Evidence code
df["Evidence"] = "IEA"
# 8O With (or) From
df["With"] = ""
# 9R Aspect
df["Aspect"] = df["GO_ID"].apply(lambda t: go[t].aspect)
# 10O DB Object Name
df["DB_Object_Name"] = ""
# 11O DB Object Synonym (|Synonym)
df["Synonym"] = ""
# 12R DB Object Type
df["DB_Object_Type"] = "protein"
# 13R Taxon (|taxon)
df["Taxon_ID"] = 'Viridiplantae'
# 14R Date
df["Date"] = str(datetime.now()).split(' ')[0]
# 15R Assigned by
df["Assigned_By"] = df["DB"]
# 16O Annotation Extension
df["Annotation_Extension"] = ""
# 17O Gene Product Form ID
df["Gene_Product_Form_ID"] = ""

In [19]:
df = df[GAF20FIELDS]

In [20]:
import gzip

with gzip.open('../data/functional_predictions/all_predictions.gaf.gz', 'wt') as fp:
    print('!gaf-version: 2.0', file=fp)
    for i in tqdm(range(0,len(df),int(1e5))):
        df.iloc[i:i+int(1e5)].to_csv(fp, sep='\t', index=False, header=False)

  0%|          | 0/673 [00:00<?, ?it/s]

In [21]:
len(df)

67296198

filter the terms to quickgo validated terms

In [4]:
with open('../data/quickgo/viridiplantae/valid_terms.txt', 'rt' ) as fp:
    vir_go_ts = set(map(lambda x: x.rstrip(), fp.readlines()))

def expand_terms(ts):
    z = set()
    for t in ts:
        if t in go:
            z.add(t)
            z.add(str(go[t]))
            for t1 in go[t].parentsR:
                z.add(str(go[t1]))
    return z

In [5]:
viridiplantae_terms = expand_terms(vir_go_ts)

In [7]:
'GO:0007420' in viridiplantae_terms

True

In [15]:
for t in vir_go_ts:
    if t in go and 7420 in go[t].parentsR:
        print(t)

GO:0022027


In [17]:
go[22027].name

'interkinetic nuclear migration'

In [25]:
import gzip

with gzip.open('../data/functional_predictions/all_predictions_filtered_viridiplantae.gaf.gz', 'wt') as fp:
    print('!gaf-version: 2.0', file=fp)
    zdf = df[df.GO_ID.isin(viridiplantae_terms)]
    for i in tqdm(range(0,len(zdf),int(1e5))):
        zdf.iloc[i:i+int(1e5)].to_csv(fp, sep='\t', index=False, header=False)

  0%|          | 0/638 [00:00<?, ?it/s]